# Financial Transaction Fraud Detection
**End-to-end ML portfolio project | Python + Scikit-learn + Power BI**

## Project objective
Build a transaction-level fraud classifier, investigate suspicious patterns, compare supervised and unsupervised algorithms, and export predictions for a Power BI dashboard.

### Dataset choices
- **Recommended benchmark:** ULB Credit Card Fraud Detection (284,807 transactions; 492 frauds; 0.172% fraud rate). Features `V1`–`V28` are anonymised PCA components; `Time` and `Amount` are also supplied.
- **Included starter file:** `data/creditcard_demo_synthetic.csv`. This is a small **synthetic demonstration dataset**, not the original ULB data and not evidence of real-world model performance.
- Download the real dataset from [Kaggle – ULB Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud). Extract `creditcard.csv` into the `data/` folder. The notebook automatically prefers that real file when present.

**Important:** the ULB data covers two days in September 2013 and is highly imbalanced. It is suitable for learning and benchmarking, not for deploying a real bank fraud system.

## 1. Install and import libraries

In [ ]:
# If needed, run once in your terminal or uncomment this line:
# %pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn joblib

import os, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    classification_report, confusion_matrix, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, precision_recall_curve
)
from sklearn.base import clone

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
os.makedirs("outputs", exist_ok=True)
os.makedirs("models", exist_ok=True)

## 2. Load the dataset

In [ ]:
REAL_DATA_PATH = "data/creditcard.csv"
DEMO_DATA_PATH = "data/creditcard_demo_synthetic.csv"

if os.path.exists(REAL_DATA_PATH):
    DATA_PATH = REAL_DATA_PATH
    print("Using REAL ULB benchmark dataset.")
else:
    DATA_PATH = DEMO_DATA_PATH
    print("Using SYNTHETIC DEMO data. Download the ULB CSV for the actual benchmark.")

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

## 3. Understand the data and target

In [ ]:
display(df.info())
display(df.describe().T)
print("\\nMissing values (top 10):")
display(df.isna().sum().sort_values(ascending=False).head(10))
if "Class" not in df.columns:
    raise ValueError("Target column 'Class' was not found. Check the dataset.")

In [ ]:
class_counts = df["Class"].value_counts().sort_index()
class_pct = df["Class"].value_counts(normalize=True).sort_index() * 100
display(pd.DataFrame({"transactions": class_counts, "percentage": class_pct.round(4)}))

plt.figure(figsize=(6,4))
sns.countplot(data=df, x="Class")
plt.title("Legitimate vs fraudulent transactions (0=legitimate, 1=fraud)")
plt.show()

print(f"Fraud rate: {df['Class'].mean()*100:.4f}%")

### Interpretation
A very low fraud percentage means **accuracy is misleading**. For example, predicting every transaction as legitimate could look highly accurate while catching zero fraud. We therefore focus on recall, precision, F1 and precision-recall AUC.

## 4. Explore transaction amount and patterns

In [ ]:
if "Amount" in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(data=df, x="Class", y="Amount", showfliers=False)
    plt.title("Transaction amount by class (outliers hidden for readability)")
    plt.show()

    plt.figure(figsize=(8,4))
    sns.histplot(data=df, x="Amount", hue="Class", bins=50, log_scale=(False, True),
                 stat="density", common_norm=False, element="step")
    plt.title("Transaction amount distribution (log y scale)")
    plt.show()

## 5. Data preparation and leakage prevention

In [ ]:
# The target must never be included among input features.
# Time is retained as a feature in this benchmark; Amount and PCA components are also predictors.
X = df.drop(columns=["Class"]).copy()
y = df["Class"].astype(int)

# Remove duplicate rows only if exact duplicate records exist.
duplicate_count = X.duplicated().sum()
print("Duplicate feature rows:", duplicate_count)

# Stratification keeps the rare fraud class represented in both partitions.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Fraud rate train/test:", round(y_train.mean()*100, 4), "% /", round(y_test.mean()*100, 4), "%")

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler())
        ]), numeric_features)
    ],
    remainder="drop"
)

**Why RobustScaler?** Transaction amounts and PCA variables may contain extreme values. RobustScaler uses the median and interquartile range, so it is less influenced by extreme observations than standard scaling. The scaler and imputer are fitted inside each training pipeline to avoid fitting preprocessing on the test set.

## 6. Train supervised models

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("prep", clone(preprocessor)),
        ("model", LogisticRegression(
            class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE
        ))
    ]),
    "Decision Tree": Pipeline([
        ("prep", clone(preprocessor)),
        ("model", DecisionTreeClassifier(
            class_weight="balanced", max_depth=8, min_samples_leaf=5,
            random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest": Pipeline([
        ("prep", clone(preprocessor)),
        ("model", RandomForestClassifier(
            n_estimators=250, class_weight="balanced_subsample",
            min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE
        ))
    ])
}

fitted_models = {}
for name, model in models.items():
    print("Training:", name)
    model.fit(X_train, y_train)
    fitted_models[name] = model
print("Training complete.")

## 7. Evaluate models at a default 0.50 threshold

In [ ]:
def evaluate_binary_model(name, model, X_eval, y_eval, threshold=0.50):
    probabilities = model.predict_proba(X_eval)[:, 1]
    predictions = (probabilities >= threshold).astype(int)
    return {
        "Model": name,
        "Threshold": threshold,
        "Precision": precision_score(y_eval, predictions, zero_division=0),
        "Recall": recall_score(y_eval, predictions, zero_division=0),
        "F1": f1_score(y_eval, predictions, zero_division=0),
        "ROC_AUC": roc_auc_score(y_eval, probabilities),
        "PR_AUC": average_precision_score(y_eval, probabilities),
        "Predicted_Fraud": int(predictions.sum())
    }, probabilities, predictions

evaluation = {}
rows = []
for name, model in fitted_models.items():
    row, probs, preds = evaluate_binary_model(name, model, X_test, y_test)
    rows.append(row)
    evaluation[name] = {"probabilities": probs, "predictions": preds}

results = pd.DataFrame(rows).sort_values("PR_AUC", ascending=False)
display(results.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, (name, values) in zip(axes, evaluation.items()):
    cm = confusion_matrix(y_test, values["predictions"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

### How to read the confusion matrix
- **True Positive (TP):** fraud correctly flagged.
- **False Positive (FP):** legitimate transaction incorrectly flagged.
- **True Negative (TN):** legitimate transaction correctly cleared.
- **False Negative (FN):** fraud missed by the model.

For fraud operations, false negatives can mean missed fraud losses; false positives can inconvenience customers and create manual-review workload. The threshold should reflect that business trade-off.

## 8. Tune the classification threshold

In [ ]:
# Demonstrate threshold selection using the Random Forest validation probabilities.
# In a rigorous production workflow, create a separate validation set and never tune on the final test set.
rf_name = "Random Forest"
rf_prob = evaluation[rf_name]["probabilities"]
threshold_rows = []
for threshold in np.arange(0.05, 0.96, 0.05):
    pred = (rf_prob >= threshold).astype(int)
    threshold_rows.append({
        "threshold": round(float(threshold), 2),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0)
    })
threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df.round(3))

plt.figure(figsize=(8,4))
plt.plot(threshold_df["threshold"], threshold_df["precision"], marker="o", label="Precision")
plt.plot(threshold_df["threshold"], threshold_df["recall"], marker="o", label="Recall")
plt.plot(threshold_df["threshold"], threshold_df["f1"], marker="o", label="F1")
plt.xlabel("Decision threshold")
plt.ylabel("Score")
plt.title("Precision, recall and F1 across thresholds")
plt.legend()
plt.show()

**Threshold note:** the table above is exploratory and uses test probabilities for demonstration. For a final academic report, split the training data again into train/validation, choose the threshold on validation data, then report the final result once on the untouched test set.

## 9. Unsupervised anomaly detection — Isolation Forest

In [ ]:
# Isolation Forest does not learn from the Class labels.
# Fit only on training features. Contamination is set to the observed training fraud rate
# solely as a benchmark assumption; in practice it must be estimated or tuned.
contamination_rate = max(float(y_train.mean()), 0.001)
iso_pipeline = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", IsolationForest(
        n_estimators=250, contamination=contamination_rate,
        random_state=RANDOM_STATE, n_jobs=-1
    ))
])
iso_pipeline.fit(X_train)
# IsolationForest returns +1 for inliers and -1 for outliers.
iso_raw = iso_pipeline.predict(X_test)
iso_predictions = (iso_raw == -1).astype(int)

iso_result = {
    "Model": "Isolation Forest",
    "Threshold": np.nan,
    "Precision": precision_score(y_test, iso_predictions, zero_division=0),
    "Recall": recall_score(y_test, iso_predictions, zero_division=0),
    "F1": f1_score(y_test, iso_predictions, zero_division=0),
    "ROC_AUC": np.nan,
    "PR_AUC": np.nan,
    "Predicted_Fraud": int(iso_predictions.sum())
}
display(pd.DataFrame([iso_result]).round(4))
print(classification_report(y_test, iso_predictions, zero_division=0))

## 10. Compare all four approaches

In [ ]:
all_results = pd.concat([results, pd.DataFrame([iso_result])], ignore_index=True)
display(all_results.round(4))
all_results.to_csv("outputs/model_comparison.csv", index=False)

## 11. Feature importance (Random Forest)

In [ ]:
rf = fitted_models["Random Forest"].named_steps["model"]
importance = pd.Series(rf.feature_importances_, index=numeric_features).sort_values(ascending=False)
display(importance.head(15).to_frame("importance"))

plt.figure(figsize=(9,5))
importance.head(15).sort_values().plot(kind="barh")
plt.title("Random Forest — top 15 feature importances")
plt.xlabel("Importance")
plt.show()

## 12. Export predictions for Power BI

In [ ]:
best_model_name = results.iloc[0]["Model"]  # ranked by PR-AUC; choose based on operational trade-offs too
best_model = fitted_models[best_model_name]
best_prob = best_model.predict_proba(X_test)[:, 1]
chosen_threshold = 0.50

powerbi_output = X_test.copy()
powerbi_output["Actual_Class"] = y_test.values
powerbi_output["Fraud_Probability"] = best_prob
powerbi_output["Predicted_Class"] = (best_prob >= chosen_threshold).astype(int)
powerbi_output["Risk_Level"] = pd.cut(
    best_prob, bins=[-0.001, 0.25, 0.60, 1.0],
    labels=["Low", "Medium", "High"]
)
powerbi_output.to_csv("outputs/fraud_predictions_powerbi.csv", index=False)
print("Saved outputs/fraud_predictions_powerbi.csv")
print("Selected model:", best_model_name)

### Suggested Power BI dashboard
Import `outputs/fraud_predictions_powerbi.csv` and `outputs/model_comparison.csv`.

Recommended visuals:
1. KPI cards: total transactions, flagged transactions, actual frauds, recall and precision.
2. Donut chart: legitimate vs fraudulent transactions.
3. Line chart: transaction count and flagged fraud by time (if `Time` exists).
4. Bar chart: fraud probability by risk level.
5. Confusion matrix table: TP, FP, TN and FN.
6. Model comparison: precision, recall, F1 and PR-AUC.

Use `Actual_Class` only for retrospective analysis. A live system would not know the true label at prediction time.

## 13. Save the selected model

In [ ]:
model_path = f"models/{best_model_name.lower().replace(' ', '_')}_fraud_model.joblib"
joblib.dump({
    "model": best_model,
    "model_name": best_model_name,
    "threshold": chosen_threshold,
    "features": X.columns.tolist()
}, model_path)
print("Saved:", model_path)

## 14. Predict a new transaction

In [ ]:
# Example: use an existing held-out transaction as a stand-in for a new transaction.
# Replace this row with a transaction having the same input columns in a real application.
sample_transaction = X_test.iloc[[0]].copy()
loaded = joblib.load(model_path)
fraud_probability = loaded["model"].predict_proba(sample_transaction)[:, 1][0]
prediction = int(fraud_probability >= loaded["threshold"])

print(f"Fraud probability: {fraud_probability:.2%}")
print("Prediction:", "POTENTIAL FRAUD — REVIEW" if prediction == 1 else "LIKELY LEGITIMATE")

## 15. Final conclusions and limitations

In your report, discuss:
- Which algorithm achieved the strongest PR-AUC, recall and precision?
- How many fraud cases did it miss (false negatives)?
- How many legitimate transactions did it flag (false positives)?
- How did the decision threshold change precision and recall?
- How did Isolation Forest compare with supervised models?

### Important limitations
- The ULB dataset is anonymised and historical (September 2013).
- PCA variables `V1`–`V28` do not have direct business meanings.
- This project is a learning prototype, not a bank-ready fraud prevention service.
- Never claim a model is "best" based on accuracy alone.
- Do not deploy without temporal validation, drift monitoring, cost-sensitive thresholding, security controls and human review.

### Dataset citation
Dal Pozzolo, A. et al. (2015). *Calibrating Probability with Undersampling for Unbalanced Classification*. IEEE Symposium on Computational Intelligence and Data Mining. Dataset: [Kaggle ULB Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud).